# 02 Operation and systems

This notebook shows how a building is used and serviced: its operating profile, its
domestic hot water (DHW) demand, its ventilation template and its window-to-wall ratios.
These definitions are exported from `epsm_data` and work without the catalogue.

**Prerequisites:** `epsm_data` installed. Runs offline. Energy is in kWh per m² Atemp
(heated floor area) per year; airflow in l/(s m²).

Import the profile and system functions.

In [ ]:
from epsm_data import (
    facade_assumptions,
    select_dhw_profile,
    select_operating_profile,
    ventilation_template_id,
    ventilation_templates,
)

Sveby normal use is the default: the agreed assumptions for energy calculations.

In [ ]:
profile = select_operating_profile("multi_family")
use = profile.values["use"]
print(profile)
print("Heating setpoint:", use["heating_setpoint_c"], "°C")
print("DHW:", use["dhw_kwh_m2_year"], "kWh/(m² year)")
print("Household electricity:", use["household_electricity_kwh_m2_year"], "kWh/(m² year)")

OperatingProfile(id='sveby.residential.v2.0.2024-11-11', inputs=17, sources=1)
Heating setpoint: 21.0 °C
DHW: 21.0 kWh/(m² year)
Household electricity: 21.0 kWh/(m² year)


`standard="measured"` replaces Sveby values with measured operation of the Swedish
stock, for comparing simulations with metered energy. For multi-family buildings the
measured heating setpoint depends on the era, so `era` is required.

In [ ]:
for era in ("SE.01", "SE.02", "SE.03", "SE.04", "SE.05", "SE.06", "SE.07"):
    measured = select_operating_profile("multi_family", standard="measured", era=era)
    print(era, measured.values["use"]["heating_setpoint_c"], "°C")

SE.01 22.2 °C
SE.02 22.6 °C
SE.03 22.5 °C
SE.04 22.4 °C
SE.05 23.0 °C
SE.06 23.0 °C
SE.07 23.0 °C


The measured profile keeps the Sveby schema and records exactly which values it replaced.

In [ ]:
measured = select_operating_profile("multi_family", standard="measured", era="SE.02")
print(measured.profile_id)
print("Changed:", measured.values["measured_use"]["changed"])

epsm.measured_use.v1.multi_family.SE.02
Changed: ('use.heating_setpoint_c',)


For offices the measured values change the ventilation airflow and running hours.

In [ ]:
sveby = select_operating_profile("office")
measured = select_operating_profile("office", standard="measured", era="SE.03")
for label, record in (("Sveby", sveby), ("Measured", measured)):
    operation = record.values["operation_inputs"]
    print(
        f"{label:<9} {record.values['use']['mechanical_vent_l_s_m2']} l/(s m²), "
        f"from {operation['ventilation_start_hour']}:00 for {operation['ventilation_hours']} h"
    )

Sveby     1.3 l/(s m²), from 7:00 for 12 h
Measured  1.6 l/(s m²), from 6:00 for 14 h


The BEN normal-use DHW coefficient is a separate, explicit choice for energy
performance certificate (EPC) comparisons. It does not change the operating profile.

In [ ]:
for building_type in ("single_family", "multi_family", "office", "school"):
    dhw = select_dhw_profile(building_type, purpose="epc_normal_use")
    print(f"{building_type:<14} {dhw.values['dhw_kwh_m2_year']:5.1f} kWh/(m² {dhw.values['area_basis']} year)")

single_family   20.0 kWh/(m² Atemp year)
multi_family    25.0 kWh/(m² Atemp year)
office           2.0 kWh/(m² Atemp year)
school           2.0 kWh/(m² Atemp year)


Ventilation templates describe the system type: self-draught (S), exhaust (F),
supply and exhaust (FT) and heat recovery (FTX) at three recovery effectivenesses.

In [ ]:
for template_id, template in ventilation_templates().items():
    print(f"{template_id:<26} {dict(template.values['airflow'])}")

epsm.ventilation.S         {'topology': 'S', 'mechanical_vent_l_s_m2': 0, 'natural_exchange_ach': 0.5}
epsm.ventilation.F         {'topology': 'F', 'sensible_recovery_effectiveness': 0}
epsm.ventilation.FT        {'topology': 'FT', 'sensible_recovery_effectiveness': 0}
epsm.ventilation.FTX_060   {'topology': 'FTX', 'sensible_recovery_effectiveness': 0.6}
epsm.ventilation.FTX_075   {'topology': 'FTX', 'sensible_recovery_effectiveness': 0.75}
epsm.ventilation.FTX_080   {'topology': 'FTX', 'sensible_recovery_effectiveness': 0.8}


Without an observed system the era default applies. If you know the installed system,
for example from the EPC, pass it as `ventilation`.

In [ ]:
for system in (None, "S", "F", "FT", "FTX"):
    template_id = ventilation_template_id("multi_family", "SE.02", "existing", system)
    print(f"{system or 'era default':<12} -> {template_id}")

era default  -> epsm.ventilation.F
S            -> epsm.ventilation.S
F            -> epsm.ventilation.F
FT           -> epsm.ventilation.FT
FTX          -> epsm.ventilation.FTX_075


Renovation overrides the observed system: standard keeps it, advanced installs FTX at 0.80.

In [ ]:
for state in ("existing", "standard", "advanced"):
    print(f"{state:<9} -> {ventilation_template_id('multi_family', 'SE.02', state, 'F')}")

existing  -> epsm.ventilation.F
standard  -> epsm.ventilation.F
advanced  -> epsm.ventilation.FTX_080


The catalogue applies the same rule when you select an archetype with an observed system.

In [ ]:
from epsm_data import Catalogue

catalogue = Catalogue.load()
archetype = catalogue.for_year("multi_family", 1965, state="existing", ventilation="FTX")
print(archetype.hvac_template.template_id)

epsm.ventilation.FTX_075


Facade window-to-wall ratios (WWR) fill in buildings without measured openings. Each
record says whether it is a sourced value or an EPSM assumption.

In [ ]:
for building_type in ("single_family", "multi_family", "office", "school"):
    facade = facade_assumptions(building_type)
    print(f"{building_type:<14} default {facade['default_wwr']:.2f}  {facade['wwr_by_orientation']}  ({facade['tier']})")
print(facade["orientation_rule"])

single_family  default 0.18  {'N': 0.12, 'E': 0.17, 'S': 0.26, 'W': 0.17}  (tabula)
multi_family   default 0.24  {'N': 0.22, 'E': 0.23, 'S': 0.28, 'W': 0.23}  (tabula)
office         default 0.30  {'N': 0.3, 'E': 0.3, 'S': 0.3, 'W': 0.3}  (assumption)
school         default 0.25  {'N': 0.25, 'E': 0.25, 'S': 0.25, 'W': 0.25}  (assumption)
Classify each facade by the compass direction its outward normal faces: N for 315-45 degrees, E for 45-135, S for 135-225, W for 225-315.
